# Ways to call the GTO Casida kernel

All paths build a `GTOKernel` and solve with `run_casida`. Prefer the **adapter** factories.

| Pattern | Entry point | Use when |
|---------|-------------|----------|
| A | `extract_gto_kernel(mf)` | Closed-shell singlet / triplet TDDFT |
| B | Direct `GTOKernel(...)` | Custom MO window |
| C | `extract_gto_kernel(..., tda=False)` | Full TDDFT / RPA (hybrids → dense A/B) |
| D | `extract_sf_gto_kernel(mol)` | Collinear spin-flip TDDFT |
| E | `build_spin_flip_kernel` / deprecated `GTOKernel.build_spin_flip` | Kernel only |

Molecule: H₂O (closed-shell demos) and CH₂ triplet (SF demo).

### Amarel / `libffi.so.7`

If imports fail with `libffi.so.7: cannot open shared object file`, in a terminal **before** launching / restarting the kernel:

```bash
source /projectsn/mp1009_1/am4655/casidapy/tutorials/setup_env.sh
```

Then **Restart Kernel** in the notebook.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from pyscf import gto, dft

from casidapy import (
    extract_gto_kernel,
    extract_sf_gto_kernel,
    build_spin_flip_kernel,
    run_casida,
    GTOKernel,
)
from casidapy.xas import omega_ev, plot_sticks  # spectrum helpers

HA_TO_EV = 27.211386245988

## Shared H₂O mean-field

In [ ]:
mol = gto.M(
    atom="""
    O  0.000000  0.000000  0.117300
    H  0.000000  0.757200 -0.469200
    H  0.000000 -0.757200 -0.469200
    """,
    basis="sto-3g",
    verbose=0,
)
mf = dft.RKS(mol).density_fit()
mf.xc = "pbe"
mf.kernel()
print(f"E = {mf.e_tot:.6f} Ha  nocc={int(np.sum(mf.mo_occ > 0))}")

## A — Recommended: `extract_gto_kernel` + `run_casida` (TDA singlet)

In [ ]:
kernel, opts = extract_gto_kernel(
    mf, n_states=6, n_unocc=8, tda=True, use_df=True, k_cache_max=0
)
res_a = run_casida(kernel, opts)
print("A TDA singlet (eV):", np.round(omega_ev(res_a), 2))

## B — Direct `GTOKernel` with an explicit MO window

Useful when the adapter defaults are not what you want. Indices are into the full MO set.

In [ ]:
from casidapy.casida_api import CasidaOptions

nocc = int(np.sum(mf.mo_occ > 1e-6))
occ = np.arange(max(0, nocc - 2), nocc)       # top 2 occupied
virt = np.arange(nocc, nocc + 4)              # first 4 virtuals

kernel_b = GTOKernel(
    mol,
    mf.mo_coeff,
    mf.mo_energy,
    mf.mo_occ,
    occ_indices=occ,
    virt_indices=virt,
    xc=mf.xc,
    mf=mf,
    use_df=True,
    k_cache_max=0,
)
opts_b = CasidaOptions(
    n_occ=kernel_b.n_occ,
    n_unocc=kernel_b.n_unocc,
    n_states=4,
    tda=True,
    matrix_free=True,
    basis="gto",
    xc=mf.xc,
)
res_b = run_casida(kernel_b, opts_b)
print("B custom window (eV):", np.round(omega_ev(res_b), 2))
print("  n_trans =", kernel_b.n_trans)

## C — Triplet response and hybrid full TDDFT

- `spin_state="triplet"` — triplet-adapted `f_xc` on a closed-shell reference  
- `tda=False` + hybrid XC — dense A/B via PySCF `get_ab()`

In [ ]:
kt, ot = extract_gto_kernel(
    mf, n_states=4, n_unocc=6, tda=True, spin_state="triplet", k_cache_max=0
)
res_t = run_casida(kt, ot)
print("C1 triplet TDA (eV):", np.round(omega_ev(res_t), 2))

mf_hyb = dft.RKS(mol).density_fit()
mf_hyb.xc = "pbe0"
mf_hyb.kernel()
kh, oh = extract_gto_kernel(
    mf_hyb, n_states=4, n_unocc=6, tda=False, use_df=True, k_cache_max=0
)
res_h = run_casida(kh, oh)
print("C2 PBE0 full TDDFT (eV):", np.round(omega_ev(res_h), 2))

## D — Spin-flip TDDFT (`extract_sf_gto_kernel`)

High-spin UKS reference → α-occ → β-virt manifold. Exchange-only (needs a hybrid).

In [ ]:
mol_ch2 = gto.M(
    atom="C 0 0 0; H 0 0 1.1; H 0.9 0 -0.3",
    basis="sto-3g",
    spin=2,
    verbose=0,
)
ksf, osf = extract_sf_gto_kernel(
    mol_ch2, xc="bhandhlyp", n_states=5, use_df=False
)
res_sf = run_casida(ksf, osf)
print("D SF-TDA (eV):", np.round(omega_ev(res_sf), 2))
print("  spin_flip=", ksf._spin_flip, " n_trans=", ksf.n_trans)

## E — Kernel-only SF factory (same as D without options packing)

In [ ]:
k_only = build_spin_flip_kernel(mol_ch2, xc="bhandhlyp", use_df=False, mf=ksf.mf)
# Equivalent deprecated wrapper:
# k_only = GTOKernel.build_spin_flip(mol_ch2, xc="bhandhlyp", use_df=False, mf=ksf.mf)
print("E kernel n_trans=", k_only.n_trans)

## Plot A vs C1 (singlet vs triplet sticks)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
plot_sticks(res_a, ax=ax, label="singlet TDA", color="C0", broaden_sigma_ev=0.12)
plot_sticks(res_t, ax=ax, label="triplet TDA", color="C3", broaden_sigma_ev=0.12)
ax.set_title("H₂O STO-3G / PBE")
plt.tight_layout()
plt.show()